# Đối chứng cắt cứng và phanh ngẫu nhiên trên vLLM: 500 câu GSM8K (ids 100–599), 16.384 token

Trả lời phản biện B1.1, B2.4 và B1.4. Hiện đối chứng cắt cứng và random exit mới chỉ có ở 768 token, pipeline HF, 200 câu. Notebook này chạy chúng **trên đúng pipeline vLLM và đúng 500 câu** của `gsm8k_500`, rồi suy ra cả 9 ngân sách (512 → 16.384) bằng cách cắt chuỗi như trước.

| Phương pháp | Ý nghĩa |
|---|---|
| `hardtrunc:100`, `hardtrunc:130` | Cắt cứng: ép `</think>\n` sau đúng N token suy nghĩ (giống `run_controls.py`) |
| `random` | Phanh ở biên dấu câu đầu tiên sau một bước rút ngẫu nhiên từ phân phối bước phanh của RPDI-EE16 trên chính 500 câu này (`results/gsm8k_500/rpdi_exit_ref.json`) |
| `random_matched` | Như `random` nhưng **chỉ ở 305 câu mà RPDI-EE16 đã phanh**: cùng tỷ lệ phanh, chỉ khác vị trí phanh. Tách "có phanh hay không" khỏi "phanh ở đâu" |

Câu hỏi cần trả lời: (1) cắt cứng có non-inferior ở 1.024 token như RPDI-EE16 không; (2) cắt cứng có phá được 53 vòng lặp ở 16k không; (3) `random_matched` có bằng RPDI-EE16 không.

**Phần 2 (tùy chọn, cell 9):** chạy lại `standard` và `rpdi_legacy:16:1.5` vào thư mục riêng để đo dao động giữa hai lần chạy (B1.4).

**Thời gian ước tính trên T4:** phần 1 khoảng 2–3 giờ, phần 2 khoảng 2–3 giờ. Chạy lần lượt từ trên xuống; cell 5 (smoke test) mất khoảng 5 phút, nếu lỗi thì dừng và gửi lỗi cho Claude.

**Trước khi chạy:** code mới phải đã được push lên GitHub, vì notebook clone repo từ đó.

## 1. Kiểm tra GPU và môi trường

In [ ]:
import os, subprocess
# Không dựa vào thư mục /kaggle: máy Colab cũng có sẵn thư mục này
IN_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ
print("Môi trường:", "Kaggle" if IN_KAGGLE else "Colab")
gpus = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                      capture_output=True, text=True).stdout.strip()
print(gpus or "KHÔNG THẤY GPU")
if not gpus:
    raise RuntimeError("Chưa bật GPU. Colab: Runtime → Change runtime type → T4 GPU. Kaggle: Settings → Accelerator → GPU T4 x2.")
if "P100" in gpus:
    raise RuntimeError("vLLM không chạy trên P100. Kaggle: Settings → Accelerator → GPU T4 x2.")

def vllm_busy():
    """PID các tiến trình run_vllm.py đang chạy nền (T4 không đủ bộ nhớ cho hai tiến trình vLLM)."""
    return subprocess.run(["pgrep", "-f", "vllm_engine/run_vllm.py"], capture_output=True, text=True).stdout.split()

if vllm_busy():
    print("LƯU Ý: đang có run_vllm.py chạy nền, PID", ", ".join(vllm_busy()), "- GPU đang bận.")

## 2. Lấy code mới nhất

In [ ]:
REPO_DIR = "/kaggle/working/slm-reasoning-project" if IN_KAGGLE else "/content/slm-reasoning-project"
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/luw-cid/slm-reasoning-project.git {REPO_DIR}
%cd {REPO_DIR}
!git pull
!git log --oneline -1

## 3. Cài vLLM (bản V0, có logits processor theo từng request)

vLLM 0.6.3 chỉ hỗ trợ Python ≤ 3.12, nên cell này tạo một môi trường Python 3.12 riêng ở `/tmp/vllm-env` và cho các lệnh `!python` ở những cell sau dùng môi trường đó. Môi trường có sẵn của Colab/Kaggle không bị đụng tới, nên **không cần khởi động lại runtime**. Nếu kernel bị khởi động lại, chỉ cần chạy lại cell này.

In [ ]:
# vLLM 0.6.3 (engine V0, có logits processor theo từng request) chỉ chạy trên Python <= 3.12,
# trong khi Kaggle/Colab có thể đã lên Python 3.13. Tạo môi trường Python 3.12 riêng bằng uv.
import os, subprocess
VENV = "/tmp/vllm-env"   # ngoài /kaggle/working để không bị lưu vào output của Kaggle
PYBIN = f"{VENV}/bin/python"
CHECK = ("import vllm, torch, transformers; print('vllm', vllm.__version__, '| torch', torch.__version__, "
         "'| transformers', transformers.__version__, '| CUDA', torch.cuda.is_available())")

if not (os.path.exists(PYBIN) and subprocess.run([PYBIN, "-c", "import vllm"], capture_output=True).returncode == 0):
    print("Đang cài môi trường Python 3.12 + vLLM 0.6.3 (khoảng 3–6 phút)...")
    !pip install -q uv
    !python -m uv venv {VENV} --python 3.12
    !python -m uv pip install --python {PYBIN} "vllm==0.6.3.post1" "transformers==4.46.3" pytest 2>&1 | tail -n 15

# Từ đây, mọi lệnh `!python ...` ở các cell sau sẽ dùng Python của môi trường này.
# Nếu kernel bị khởi động lại, chạy lại cell này (lần sau chỉ mất vài giây).
os.environ["PATH"] = f"{VENV}/bin:" + os.environ["PATH"]
check = subprocess.run(["python", "-c", CHECK], capture_output=True, text=True)
print(check.stdout or check.stderr[-3000:])
if check.returncode != 0:
    raise RuntimeError("Cài vLLM thất bại. Gửi toàn bộ output của cell này cho Claude.")
!python -m pytest tests -q

## 4. Thư mục kết quả

In [ ]:
if IN_KAGGLE:
    OUT_ROOT = "/kaggle/working/vllm_runs"
else:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_ROOT = "/content/drive/MyDrive/slm_rlvr/vllm_runs"

START, END = 100, 600
MAIN_DIR = f"{OUT_ROOT}/gsm8k_500"             # lần chạy chính đã có (standard, RPDI-EE16, RPDI-EE512)
CTRL_DIR = f"{OUT_ROOT}/gsm8k_500_controls"    # phần 1
REPEAT_DIR = f"{OUT_ROOT}/gsm8k_500_repeat"    # phần 2 (tùy chọn)
SMOKE_DIR = f"{OUT_ROOT}/smoke_controls"
CTRL_METHODS = "hardtrunc:100,hardtrunc:130,random,random_matched"
REPEAT_METHODS = "standard,rpdi_legacy:16:1.5"
DELTA = 3.0  # biên δ đã chốt trong bài, KHÔNG đổi

MAIN_BUDGETS = f"{MAIN_DIR}/budgets.jsonl"
if not os.path.exists(MAIN_BUDGETS):
    MAIN_BUDGETS = "results/gsm8k_500/budgets.jsonl"
print("Kết quả lần chạy chính:", MAIN_BUDGETS,
      "(có)" if os.path.exists(MAIN_BUDGETS) else "(THIẾU: chép budgets.jsonl của gsm8k_500 lên Drive)")
for d in (CTRL_DIR, REPEAT_DIR):
    os.makedirs(d, exist_ok=True)
print("Kết quả mới sẽ lưu tại:", OUT_ROOT)

## 5. Smoke test: 3 câu (ids 100–102), ngân sách 1.024

In [ ]:
import json
if vllm_busy():
    raise RuntimeError(f"GPU đang bận (PID {', '.join(vllm_busy())}). Đợi lần chạy đó xong rồi mới chạy smoke test.")
!rm -rf {SMOKE_DIR}
!python -u method_c_rpdi_ee/vllm_engine/run_vllm.py --start 100 --end 103 --max_tokens 1024 --budgets 512,768,1024 --methods {CTRL_METHODS} --output_dir {SMOKE_DIR}

ref = {int(k): v for k, v in json.load(open("results/gsm8k_500/rpdi_exit_ref.json", encoding="utf-8"))["exit_steps"].items()}
raws = [json.loads(l) for l in open(f"{SMOKE_DIR}/raw.jsonl", encoding="utf-8")]
assert len(raws) == 12, f"Cần 12 dòng, có {len(raws)}"
for r in raws:
    print(f"id={r['sample_id']} {r['method']:<16} token={len(r['output_ids']):>5} </think>@{r['close_index']} "
          f"phanh={r['forced_close']} exit_step={r['exit_step']} hẹn={r.get('scheduled_exit_at')}")
    if r["method"].startswith("hardtrunc") and r["forced_close"]:
        assert r["exit_step"] == int(r["method"].split("N")[1]) == r["close_index"], r
    if r["method"] == "random_matched" and ref[r["sample_id"]] < 0:
        assert not r["forced_close"], "random_matched không được phanh ở câu RPDI-EE16 không phanh"
print("\nSMOKE TEST ĐẠT")

## 6. Phần 1: chạy 4 đối chứng trên 500 câu ở 16.384 token (chạy nền)

Chạy bằng `nohup`. Nếu runtime bị ngắt: chạy lại các cell 1–4 rồi cell này; script tự bỏ qua các lượt đã lưu.

In [ ]:
def launch(methods, out_dir):
    if vllm_busy():
        raise RuntimeError(f"Đã có run_vllm.py đang chạy (PID {', '.join(vllm_busy())}). Theo dõi bằng cell 7.")
    cmd = (f"nohup python -u method_c_rpdi_ee/vllm_engine/run_vllm.py --start {START} --end {END} --max_tokens 16384 "
           f"--methods {methods} --output_dir {out_dir} >> {out_dir}/log.txt 2>&1 &")
    print(cmd)
    os.system(cmd)

launch(CTRL_METHODS, CTRL_DIR)

## 6b. Sao lưu kết quả sang Google Drive (Colab)

Chạy ngay sau cell 6. Cứ 5 phút sao chép thư mục kết quả sang Drive một lần, chạy nền và không ảnh hưởng tiến trình sinh. Nếu kết quả đã ghi thẳng lên Drive (cell 4 in ra đường dẫn `/content/drive/...`) hoặc đang chạy trên Kaggle thì cell này tự bỏ qua.

In [ ]:
BACKUP = "/content/drive/MyDrive/slm_rlvr/vllm_runs"
if IN_KAGGLE:
    print("Kaggle: không có Drive. Khi chạy xong, tải kết quả từ panel Output.")
elif OUT_ROOT.startswith("/content/drive"):
    print("Kết quả đã ghi thẳng lên Drive:", OUT_ROOT, "-> không cần sao lưu.")
else:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(BACKUP, exist_ok=True)
    os.system(f"nohup bash -c 'while true; do cp -r {OUT_ROOT}/. {BACKUP}/; sleep 300; done' > /dev/null 2>&1 &")
    print(f"Đang sao lưu {OUT_ROOT} -> {BACKUP} mỗi 5 phút")

## 7. Theo dõi tiến độ (chạy lại bao nhiêu lần cũng được)

Khi đang chạy phần 2, đổi dòng đầu thành `WATCH, WATCH_METHODS = REPEAT_DIR, REPEAT_METHODS`.

In [ ]:
import re
from collections import Counter
WATCH, WATCH_METHODS = CTRL_DIR, CTRL_METHODS
TOTAL = (END - START) * len(WATCH_METHODS.split(","))
raw_path, log_path = f"{WATCH}/raw.jsonl", f"{WATCH}/log.txt"
saved = 0
if os.path.exists(raw_path):
    saved = sum(1 for line in open(raw_path, encoding="utf-8") if line.startswith('{"sample_id"'))
log = open(log_path, encoding="utf-8", errors="ignore").read() if os.path.exists(log_path) else ""
done_lines = re.findall(r"^\[\s*\d+/\d+\] id=.*$", log, flags=re.M)
beats = re.findall(r"\[đang sinh\] (\d+) lượt chưa xong, dài nhất (\d+) token", log)
print(f"Tiến trình run_vllm.py: {'ĐANG CHẠY' if vllm_busy() else 'không chạy'}")
print(f"Đã xong  : {saved}/{TOTAL} lượt ({100 * saved / TOTAL:.0f}%)")
if beats and vllm_busy():
    print(f"Đang sinh: {beats[-1][0]} lượt, lượt dài nhất đã tới {int(beats[-1][1]):,} / 16,384 token")
if done_lines:
    print("\n10 lượt vừa xong:\n" + "\n".join(done_lines[-10:]))
if "Hoàn tất" in log:
    print("\nĐÃ XONG")
!nvidia-smi --query-gpu=utilization.gpu,memory.used --format=csv,noheader

## 8. Phân tích phần 1 (khi đủ 2000 lượt)

So RPDI-EE16 với từng đối chứng ở 768 và 1.024 token, và so mỗi đối chứng (ở 1.024 và 16.384 token) với `standard` ở 16.384 token, cùng biên δ = 3 như bài. Cuối cell là số lượt chạm trần 16k (vòng lặp).

In [ ]:
R = "rpdi_legacy_W16_l1.5"
CTRL = ["hardtrunc_N100", "hardtrunc_N130", "random", "random_matched"]
pairs = [f"{R}@{b}:{c}@{b}" for b in (768, 1024) for c in CTRL]
pairs += [f"{c}@{b}:standard@16384" for b in (1024, 16384) for c in CTRL]
PAIRS = " ".join(pairs)
!python method_c_rpdi_ee/vllm_engine/analyze_budgets.py --budgets_file {MAIN_BUDGETS} {CTRL_DIR}/budgets.jsonl --compare {PAIRS} --delta {DELTA} --output_md {CTRL_DIR}/analysis.md --output_json {CTRL_DIR}/analysis.json > /dev/null
from IPython.display import Markdown, display
display(Markdown(open(f"{CTRL_DIR}/analysis.md", encoding="utf-8").read()))

recs = [json.loads(l) for p in (MAIN_BUDGETS, f"{CTRL_DIR}/budgets.jsonl") for l in open(p, encoding="utf-8")]
for m in ["standard", R] + CTRL:
    rs = [r for r in recs if r["method"] == m and r["max_new_tokens"] == 16384]
    if rs:
        print(f"{m:<22} n={len(rs)} chạm trần 16k: {sum(r['hit_budget'] for r in rs):>3} | "
              f"phanh: {sum(r['early_exited'] for r in rs):>3} | token TB: {sum(r['total_tokens'] for r in rs) / len(rs):,.0f}")

## 9. Phần 2 (tùy chọn): chạy lại Standard và RPDI-EE16 để đo dao động giữa hai lần chạy

Chỉ chạy khi phần 1 đã xong (T4 không chạy được hai tiến trình vLLM cùng lúc). Cùng lệnh, cùng tham số với lần chạy chính, nên sai khác giữa hai lần phản ánh nhiễu số học fp16 khi vLLM gộp lô khác nhau.

In [ ]:
launch(REPEAT_METHODS, REPEAT_DIR)

## 10. Phân tích phần 2

In [ ]:
main = {}
for l in open(MAIN_BUDGETS, encoding="utf-8"):
    r = json.loads(l)
    main[(r["sample_id"], r["method"], r["max_new_tokens"])] = r
rows = []
for l in open(f"{REPEAT_DIR}/budgets.jsonl", encoding="utf-8"):
    r = json.loads(l)
    a = main.get((r["sample_id"], r["method"], r["max_new_tokens"]))
    if a:
        rows.append((r["method"], r["max_new_tokens"], a, r))
print(f"{'phương pháp':<22}{'ngân sách':>9}{'n':>5}{'acc lần 1':>11}{'acc lần 2':>11}{'trùng văn bản':>15}{'đổi đúng/sai':>14}")
for m in ("standard", "rpdi_legacy_W16_l1.5"):
    for b in (512, 768, 1024, 1536, 2048, 3072, 4096, 8192, 16384):
        rs = [(a, r) for mm, bb, a, r in rows if mm == m and bb == b]
        if not rs:
            continue
        n = len(rs)
        print(f"{m:<22}{b:>9}{n:>5}"
              f"{100 * sum(a['is_correct'] for a, _ in rs) / n:>10.1f}%"
              f"{100 * sum(r['is_correct'] for _, r in rs) / n:>10.1f}%"
              f"{100 * sum(a['response_sha1'] == r['response_sha1'] for a, r in rs) / n:>14.1f}%"
              f"{sum(a['is_correct'] != r['is_correct'] for a, r in rs):>14}")

## 11. Gửi lại kết quả

Tải từ Drive (`MyDrive/slm_rlvr/vllm_runs/`) và chép vào repo:

- `gsm8k_500_controls/` → `results/gsm8k_500_controls/`: `analysis.md`, `analysis.json`, `budgets.jsonl`, `run_info.json`, `log.txt`
- `gsm8k_500_repeat/` → `results/gsm8k_500_repeat/`: `budgets.jsonl`, `run_info.json`, `log.txt` (nếu đã chạy phần 2)

`raw.jsonl` lớn, cứ để trên Drive.